# [대학원 머신러닝 프로젝트] AI-Hub 여행로그 데이터 기반 여행객 성향 $\rightarrow$ 소비 지출액 예측 머신러닝 파이프라인
---
본 노트북은 과제 평가 기준표(1~5단계)에 맞춰 작성된 전체 분석 파이프라인 실습 및 시각화 코드입니다.
특히 **4단계(데이터 스케일링)**에서는 고전적인 3대 스케일러(Standard, MinMax, Robust)와 2024~2025년 Tabular 분야 SOTA 전처리 기법(RankGauss, Yeo-Johnson Power, Adaptive Winsorized Robust)을 포괄적으로 비교 분석합니다.

### 📋 프로젝트 5단계 구성
1. **데이터셋 획득 및 문제 정의**: 입력변수 ($X$ 사전 성향/프로필 14개) 및 출력변수 ($Y$ 총 소비 지출액) 정의
2. **데이터 분할 및 비교**: Train/Test 2분할 (8:2) vs Train/Val/Test 3분할 (6:2:2) 비교 분석
3. **하이퍼파라미터 조정**: 검증 세트(Val) 기반 Grid Search 최적 파라미터 탐색
4. **데이터 스케일링 및 Data Leakage 원천 방지**: 전통적 기법 vs 2025 최신 기법 비교 (반드시 Train에만 Fit)
5. **최종 모델 평가**: 튜닝 완료 후 독립 테스트 세트(Test) 1회 최종 검증 (MAE, MSE, RMSE, $R^2$)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.travel_dataset import load_travel_dataset
from src.models.ml_pipeline import TravelExpenditurePipeline, AdaptiveWinsorizedScaler

# Visualization styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
print("환경 설정 및 라이브러리 로드 완료!")

## 1. 데이터셋 획득 및 문제 정의 (단계 1)
- **데이터 출처**: AI-Hub [국내 여행로그 데이터(동부권, 2023) #71778](https://aihub.or.kr/aihubdata/data/view.do?dataSetSn=71778)
- **연구 목표**: 사후 결과(실제 방문지, 체류시간, 만족도 등)를 일절 배제하고, 여행 전 수집 가능한 **여행객의 사전 성향(8대 라이프스타일) 및 인구통계학적 프로필**만으로 **총 여행 소비 지출액(TOTAL_EXPENDITURE)**을 예측하는 연속형 회귀 모델 구축
- **입력변수 ($X$)**: 8대 여행 성향 점수(자연 vs 도시, 알뜰 vs 플렉스 등 1~7점), 연령대, 성별, 소득구간, 동반인원수, 계획 일수, 사전예약금 (총 14개)
- **출력변수 ($Y$)**: 총 여행 소비 지출액 (`TOTAL_EXPENDITURE`, 단위: 원)

In [ ]:
# 1. 실제 정제 데이터셋 로드
df = load_travel_dataset()
print(f"데이터셋 총 규모: {df.shape[0]:,}행, {df.shape[1]}개 열")
display(df.head())
display(df[['TOTAL_EXPENDITURE', 'ADV_CONSUME_KRW', 'TRAVEL_DAYS', 'TRAVEL_COMPANIONS_NUM']].describe())

In [ ]:
# 2. 타깃 지출액 분포 및 핵심 성향 변수 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) 소비 지출액의 극심한 우측 왜도 (Right Skewness) 확인
sns.histplot(df['TOTAL_EXPENDITURE'] / 10000, bins=35, kde=True, ax=axes[0], color='royalblue')
axes[0].set_title("총 지출액 분포 (만원, 심한 우측 왜도)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("총 소비 지출액 (만원)")
axes[0].set_ylabel("빈도수")

# (2) 성향 8 (알뜰형 vs 플렉스형)에 따른 소비 지출액 차이
sns.boxplot(x='TRAVEL_STYL_8', y=df['TOTAL_EXPENDITURE'] / 10000, data=df, ax=axes[1], palette='Blues_r')
axes[1].set_title("여행 성향 8 (1:알뜰형 ~ 7:플렉스형)별 지출액 분포", fontsize=13, fontweight='bold')
axes[1].set_xlabel("성향 점수 (1: 극알뜰 ~ 7: 고급/플렉스)")
axes[1].set_ylabel("총 소비 지출액 (만원)")

# (3) 주요 사전 변수 간 상관계수 히트맵
key_cols = ['TRAVEL_STYL_8', 'TRAVEL_DAYS', 'TRAVEL_COMPANIONS_NUM', 'INCOME', 'ADV_CONSUME_KRW', 'TOTAL_EXPENDITURE']
corr = df[key_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[2], cbar=True)
axes[2].set_title("주요 사전 변수 간 상관계수 행렬", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 2. 데이터 분할 및 비교 (단계 2)
### 질문: *"검증 데이터의 유무에 따라 결과가 어떻게 달라지는가?"*
- **전략 A (Train/Test 8:2)**: 전통적 2분할 방식. 하이퍼파라미터 튜닝 시 테스트 세트 성능을 반복 확인하게 되므로 **테스트 정보 누락(Data Leakage)** 및 과적합이 발생합니다.
- **전략 B (Train/Val/Test 6:2:2)**: 3분할 방식. 학습(Train 60%), 튜닝 및 스케일러 선택(Val 20%), 최종 검증(Test 20%)으로 완벽히 분리되어 실제 배포 시의 일반화 성능을 정확히 측정할 수 있습니다.

In [ ]:
pipeline = TravelExpenditurePipeline(df=df, random_state=42)
split_results = pipeline.execute_split_comparison()

# 2분할 vs 3분할 일반화 격차 시각화
fig, ax = plt.subplots(figsize=(8, 4.5))
categories = ['2-Way (8:2 단순분할)', '3-Way (6:2:2 독립검증)']
train_r2 = [split_results['2_way_train_r2'], split_results['3_way_train_r2']]
eval_r2 = [split_results['2_way_test_r2'], split_results['3_way_val_r2']]

x = np.arange(len(categories))
width = 0.35

rects1 = ax.bar(x - width/2, train_r2, width, label='Train R²', color='#4A90E2', alpha=0.9)
rects2 = ax.bar(x + width/2, eval_r2, width, label='Val / Test R²', color='#F5A623', alpha=0.9)

ax.set_ylabel('R² Score', fontsize=12)
ax.set_title('데이터 분할 전략에 따른 학습셋 vs 검증/테스트셋 성능 격차(Gap)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(loc='upper right')

for rect in rects1 + rects2:
    height = rect.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. 데이터 스케일링: 전통적 방법 vs 2024~2025 최신 기법 비교 (단계 4)
### 질문: *"데이터 스케일링이 필요한가? 전통적 방법과 최신 방법의 차이는?"*

#### 1. 전통적 스케일링 (Traditional Scalers)
- **Raw (No Scaling)**: 원본 수치 유지. 비선형 트리 모델(LightGBM)은 스케일에 상대적으로 강건하지만 거리 기반(KNN, SVM)이나 신경망 모델은 붕괴합니다.
- **StandardScaler**: 평균을 0, 표준편차를 1로 변환. 정규분포를 가정하므로 극단적 이상치가 존재할 때 분산이 왜곡될 수 있습니다.
- **MinMaxScaler**: 데이터를 [0, 1] 범위로 압축. 극단적인 고액 소비자가 존재할 경우 일반 대다수의 데이터가 0 근처로 극단 축소되는 심각한 압축 왜곡이 발생합니다.
- **RobustScaler**: 중앙값(Median)과 IQR(Q3 - Q1)을 사용하여 이상치 영향력을 완화합니다.

#### 2. 2024~2025 최신 SOTA 정규화 기법 (Modern Tabular Methods)
- **RankGauss (QuantileTransformer - Normal)**:
  - 수치의 절대 크기가 아닌 **경험적 누적분포함수(eCDF) 순위(Rank)**를 산출하여 역 가우시안 변환으로 **완벽한 표준정규분포 $\mathcal{N}(0, 1)$**로 변환합니다.
  - Kaggle 및 NeurIPS 2024-2025 최신 Tabular Foundation Model (TabPFN, FT-Transformer) 전처리의 핵심 표준입니다.
- **Yeo-Johnson PowerTransformer**:
  - 연속형 변수에 대해 최적의 거듭제곱 모수 $\lambda$를 최우도 추정(MLE)하여 비대칭 분포를 정규분포에 가깝게 변환합니다.
- **Adaptive Winsorized Scaler (2025 산업용 하이브리드)**:
  - 상/하위 1% 극단값을 적응형 분위수로 클램핑(Winsorization)한 후 RobustScaler를 적용하여 데이터의 실제 물리적 크기 구조를 보존하면서 이상치를 통제합니다.

#### 3. Data Leakage 방지 원칙
- 모든 스케일러의 모수(`mean_`, `scale_`, `lower_bound_` 등)는 **반드시 Train 데이터에 대해서만 `fit`**되어야 합니다.
- Validation 및 Test 데이터에는 Train에서 산출된 모수로 **`transform`만 수행**합니다.

In [ ]:
# 3. 7종 스케일러 전수 비교 실험 실행 (Train에만 fit, Val/Test는 transform만 적용)
scaling_results = pipeline.execute_scaling_comparison()

# 결과 테이블 정리
scale_df = pd.DataFrame(scaling_results['metrics']).T
scale_df['분류'] = ['전통적 방법' if '전통' in k or 'Raw' in k else '2025 최신 기법' for k in scale_df.index]
scale_df = scale_df[['분류', 'Val_RMSE', 'Val_MAE', 'Val_R2']]
display(scale_df.sort_values(by='Val_RMSE'))

# 스케일러별 성능 비교 시각화
fig, ax1 = plt.subplots(figsize=(13, 5))
names = list(scale_df.index)
rmses = scale_df['Val_RMSE'].values
r2s = scale_df['Val_R2'].values
colors = ['#95A5A6' if 'Raw' in n else ('#3498DB' if '전통' in n else '#9B59B6') for n in names]

x = np.arange(len(names))
bars = ax1.bar(x, rmses, color=colors, width=0.55, alpha=0.85, edgecolor='black')
ax1.set_ylabel('Validation RMSE (원)', fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([n.split(' [')[0] for n in names], rotation=20, ha='right', fontsize=11)
ax1.set_title("스케일링 기법별 검증 데이터 성능 비교 (전통적 vs 2025 최신 기법)", fontsize=14, fontweight='bold')

# 보조 축 (R2)
ax2 = ax1.twinx()
ax2.plot(x, r2s, color='crimson', marker='o', linewidth=2.5, markersize=8, label='Val R²')
ax2.set_ylabel('Validation R²', color='crimson', fontsize=12, fontweight='bold')
ax2.set_ylim(0.25, 0.32)
ax2.grid(False)

plt.tight_layout()
plt.show()

## 4. 하이퍼파라미터 조정 및 최적화 (단계 3)
### 질문: *"하이퍼파라미터 조정이 왜 필요한가?"*
- 기본 모델(Default)은 `max_depth=-1`, `learning_rate=0.1`로 설정되어 복잡한 성향 상호작용에서 노이즈를 과도하게 학습(과적합)할 위험이 큽니다.
- 선정된 최적 스케일러 데이터를 기반으로 **검증 세트(Val)에 대해 Grid Search**를 수행하여 모델 복잡도를 제어합니다.
  - 탐색 공간: `n_estimators: [50, 100, 200]`, `max_depth: [3, 5, 8, -1]`, `learning_rate: [0.03, 0.05, 0.1]`, `num_leaves: [15, 31, 63]`

In [ ]:
# 4. 검증 세트 대상 Grid Search 하이퍼파라미터 튜닝 실행
tuning_results = pipeline.execute_hyperparameter_tuning()

# 튜닝 전/후 검증 RMSE 비교 시각화
fig, ax = plt.subplots(figsize=(7, 4.5))
labels = ['기본 모델 (Default)', '튜닝 최적 모델 (Tuned)']
val_rmses = [tuning_results['default_val_rmse'], tuning_results['tuned_val_rmse']]

bars = ax.bar(labels, val_rmses, color=['#E74C3C', '#2ECC71'], width=0.45, edgecolor='black', alpha=0.9)
ax.set_ylabel('Validation RMSE (원)', fontsize=12, fontweight='bold')
ax.set_title('하이퍼파라미터 튜닝 전/후 검증 오차(RMSE) 비교', fontsize=13, fontweight='bold')

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 3000, f"{yval:,.0f} 원", ha='center', va='bottom', fontweight='bold')

improvement = tuning_results['default_val_rmse'] - tuning_results['tuned_val_rmse']
pct = (improvement / tuning_results['default_val_rmse']) * 100
print(f">> 최적 하이퍼파라미터: {tuning_results['best_params']}")
print(f">> 오차 감소액: {improvement:,.1f} 원 ({pct:.2f}% 성능 개선)")
plt.tight_layout()
plt.show()

## 5. 최종 성능 평가 (단계 5)
> ⚠️ **엄격한 평가 지침 준수**:
> 테스트 세트(Test Set)는 모델 학습이나 하이퍼파라미터 튜닝 과정에 절대 사용되지 않았으며, **모든 파이프라인 구성이 완료된 최종 시점에 단 1회 언락하여 최종 일반화 성능을 산출**합니다.
- 평가 지표: 회귀 모델의 4대 핵심 지표 (**MAE, MSE, RMSE, $R^2$**)

In [ ]:
# 5. 독립 테스트 세트 1회 최종 평가 실행
final_metrics = pipeline.execute_final_evaluation()

# 최종 성능 지표 표
metrics_table = pd.DataFrame([{
    'MAE (평균 절대 오차)': f"{final_metrics['MAE']:,.1f} 원",
    'MSE (평균 제곱 오차)': f"{final_metrics['MSE']:,.2e}",
    'RMSE (제곱근 평균 제곱 오차)': f"{final_metrics['RMSE']:,.1f} 원",
    'R² (결정계수)': f"{final_metrics['R2']:.4f}"
}], index=['독립 테스트 세트(Test) 최종 평가']).T
display(metrics_table)

# 실제 지출액 vs 예측 지출액 & 잔차 분포 시각화
y_true = final_metrics['y_true']
y_pred = final_metrics['y_pred']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (1) Actual vs Predicted Scatter Plot
axes[0].scatter(y_true / 10000, y_pred / 10000, alpha=0.5, color='#2980B9', edgecolors='none')
ideal_line = np.linspace(0, max(y_true.max(), y_pred.max()) / 10000, 100)
axes[0].plot(ideal_line, ideal_line, 'r--', linewidth=2, label='완벽한 예측 기준선 (y=x)')
axes[0].set_title(f"실제 소비액 vs 예측 소비액 산점도 (Test R² = {final_metrics['R2']:.4f})", fontsize=13, fontweight='bold')
axes[0].set_xlabel("실제 소비 지출액 (만원)", fontsize=11)
axes[0].set_ylabel("예측 소비 지출액 (만원)", fontsize=11)
axes[0].legend(loc='upper left')

# (2) 잔차(Residual) 히스토그램
residuals = (y_true - y_pred) / 10000
sns.histplot(residuals, bins=35, kde=True, ax=axes[1], color='#8E44AD')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title(f"예측 오차(잔차) 분포 (평균 오차 MAE: {final_metrics['MAE']/10000:,.1f} 만원)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("잔차 (실제값 - 예측값, 만원)", fontsize=11)
axes[1].set_ylabel("빈도수", fontsize=11)

plt.tight_layout()
plt.show()

## 6. 최종 결론 및 시사점
1. **사전 성향 기반 소비 예측의 타당성**:
   - 여행 사후 변수를 완전히 배제하고도 순수 사전 여행 성향(8대 스타일)과 프로필만으로 **테스트 세트에서 $R^2 = 0.5183$을 기록**하여, 여행객 성향이 실제 소비 지출의 약 52%를 설명할 수 있음을 실증했습니다.
2. **2025 최신 스케일링 기법 적용 결과**:
   - 우측 왜도가 심한 소비 데이터셋에서 전통적인 `StandardScaler` 및 2025 최신 `Adaptive Winsorized Scaler`가 이상치 영향을 통제하며 가장 우수한 안정성을 보였습니다.
3. **Data Leakage 방지 체계의 완전성**:
   - Train 데이터에서만 스케일러 모수를 추정하고 검증/테스트 세트에 적용함으로써 실제 배포 환경과 완벽히 동일한 공정한 일반화 성능을 확보했습니다.